In [1]:
import cv2
import os
from ultralytics import YOLO

In [2]:
MODEL_PATH = 'runs/detect/train/weights/best.pt'
VIDEO_PATH = 'video.mp4'

In [3]:
model = YOLO(MODEL_PATH)

In [4]:
OUTPUT_FRAME_DIR = './report'

if not os.path.exists(OUTPUT_FRAME_DIR):
    os.makedirs(OUTPUT_FRAME_DIR)

In [5]:
cap = cv2.VideoCapture(VIDEO_PATH)

In [6]:
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

In [7]:
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
video_writer = cv2.VideoWriter('./report/tracked_output_video.mp4', fourcc, fps, (frame_width, frame_height))

In [8]:
while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break 

    results = model.track(frame, persist=True, conf=0.8, verbose=False) 
    annotated_frame = results[0].plot()

    if results[0].boxes.id is not None:
        
        current_time_ms = cap.get(cv2.CAP_PROP_POS_MSEC)
        current_time_sec = current_time_ms / 1000.0

        filename = f"{int(current_time_sec)}.{int((current_time_sec * 100) % 100):02d}.jpg"
        output_path = os.path.join(OUTPUT_FRAME_DIR, filename)

        cv2.imwrite(output_path, annotated_frame)

    video_writer.write(annotated_frame)

    cv2.imshow("YOLOv11 Live Tracking", annotated_frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

        
cap.release()
video_writer.release()
cv2.destroyAllWindows()